# 1. Import Libraries, Configuration Setup, and Load the Dataset

**Experiment:** `07_Evidence_hierarchical`. The whole review is encoded once. Token states are pooled into sentence representations; predicted sentence evidence probabilities determine aspect-specific attention. Each resulting aspect representation feeds both its aspect classifier and the representation fusion. The holistic head consumes the fused representations together with the `[CLS]` residual. Predicted aspect labels are parallel outputs, not inputs to the holistic head.

Input: review text only. Labels are training targets. Contextual sentence states can contain information from other sentences, and `[CLS]` supplies a direct text route; this is an evidence-supervision experiment, not exclusive evidence mediation or a causal explanation.

**Paired experiment:** run `EVIDENCE_STRENGTH = 1.0` and then `0.0`, with the same seeds, trial plan, architecture, initialization and validation selection. Only the evidence-loss coefficient changes; outputs are stored separately. This isolates evidence supervision within the sentence architecture. Compare with matched holistic-only and 04 controls before interpreting broader hierarchy gains.

Quotes are selected, nonexhaustive annotations. Unquoted sentences remain UNKNOWN unless the source explicitly says UNMENTIONED with no quotes. Classification on known targets and attention localization to known selected quotes are evaluated separately. Follow the 48 cells in order in a fresh Colab runtime.


## 1.1 Install \& Import Libraries

In [1]:
# Run before imports in a fresh Colab runtime. PyTorch is supplied by Colab.
%pip -q install transformers==4.57.6 scikit-learn pandas matplotlib seaborn nltk lightgbm statsmodels tqdm joblib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 172.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 152.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# No additional install is needed here; packages are installed in the cell above.


In [3]:
# No additional install is needed here; packages are installed in the cell above.


In [4]:
# No additional install is needed here; packages are installed in the cell above.


In [5]:
import os
import re
import gc
import json
import math
import random
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, classification_report, confusion_matrix

import time
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

def aux_hash(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

def aux_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)

def aux_validate_config(cfg):
    aspects = tuple(cfg["aspects"])
    if len(set(aspects)) != len(aspects) or any(a not in AUX_ASPECTS for a in aspects):
        raise ValueError("Unknown or repeated aspect.")
    if cfg["target"] not in ("original", "holistic", "aspects"):
        raise ValueError("Unknown training target.")
    if cfg["target"] == "original" and aspects:
        raise ValueError("Original baseline has no auxiliary heads.")
    if cfg["target"] == "aspects" and aspects != AUX_ASPECTS:
        raise ValueError("Standalone full-aspects condition must predict all four aspects.")
    if not math.isfinite(cfg["aux_strength"]) or cfg["aux_strength"] < 0:
        raise ValueError("Auxiliary strength must be finite and non-negative.")
    if cfg["target"] == "holistic" and aspects and cfg["aux_strength"] <= 0:
        raise ValueError("Joint condition needs positive auxiliary strength.")
    if cfg["n_trials"] < 1 or cfg["batch_size"] < 1 or cfg["max_length"] < 1:
        raise ValueError("Training sizes must be positive.")
    if not cfg["backbones"] or any(b not in AUX_BACKBONES for b in cfg["backbones"]):
        raise ValueError("Select ALBERT and/or BioBERT.")
    # Evidence-supervised hierarchical model: all four aspects, evidence labels aligned to sentences.
    if cfg["target"] != "holistic" or aspects != AUX_ASPECTS:
        raise ValueError("The evidence-supervised model needs the holistic target and all four aspect heads.")
    if cfg["attention_size"] < 1 or cfg["max_sentences"] < 1:
        raise ValueError("Attention size and sentence limit must be positive.")
    if not math.isfinite(cfg["evidence_strength"]) or cfg["evidence_strength"] < 0:
        raise ValueError("Evidence strength must be finite and non-negative.")


def require_finite_tensor(value, description):
    if not torch.isfinite(value).all():
        raise FloatingPointError(f"Non-finite {description}; no checkpoint or prediction will be selected.")

def require_finite_model(model):
    for name, value in model.state_dict().items():
        if value.is_floating_point():
            require_finite_tensor(value, f"model state {name}")

def require_probabilities(values, n_classes):
    values = np.asarray(values)
    if (values.ndim != 2 or values.shape[1] != n_classes or not np.isfinite(values).all()
            or (values < 0).any() or (values > 1).any()
            or not np.allclose(values.sum(axis=1), 1.0, atol=1e-5, rtol=1e-5)):
        raise ValueError("Invalid probability matrix.")
    return values

def checked_optimizer_step(loss, model, optimizer, scaler, max_norm, scheduler=None):
    require_finite_tensor(loss, "training loss")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm, error_if_nonfinite=not scaler.is_enabled())
    previous_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    updated = scaler.get_scale() >= previous_scale
    if updated and not torch.isfinite(grad_norm):
        raise FloatingPointError("Non-finite gradient norm without a skipped optimizer update.")
    if updated and scheduler is not None:
        scheduler.step()
    return updated

def aux_output_version(cfg):
    return f"{AUX_VERSION}_evidence_{format(cfg['evidence_strength'], '.12g').replace('.', 'p')}"


def aux_model_contract():
    """Configuration-cell constants that affect model/data meaning are recorded explicitly."""
    keys = ("AUX_BACKBONES", "AUX_ASPECTS", "AUX_STATES", "AUX_OVERALL", "AUX_SOURCE_STATES",
            "AUX_STATE_TO_CLASS", "AUX_RELEASE", "EVIDENCE_CLASSES", "EVIDENCE_IGNORE",
            "EVIDENCE_ALIGNMENT_VERSION", "UPSTREAM_MANIFEST_SHA256", "INPUT_CONDITIONS")
    # JSON normalization avoids tuple/list mismatches when a completed identity is read again.
    return json.loads(json.dumps({key: globals()[key] for key in keys if key in globals()}, sort_keys=True))


## 1.2 Seed \& Device

In [6]:

def aux_seed(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def aux_worker_seed(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}; select the experiment seed in 1.3.")
# Trial i uses RUN_SEED + i; the saved data split is independent of training seeds.


Device: cuda; select the experiment seed in 1.3.


## 1.3 Configuration

In [7]:
RUN_SEED = 2025
SEED = RUN_SEED
aux_seed(RUN_SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")
EXPERIMENT_FOLDER = '07_Evidence_hierarchical'
TRAINING_TARGET = 'holistic'
ASPECT_NAMES = ['efficacy', 'safety', 'burden', 'cost']
USE_METADATA = False

# Sentence level and evidence supervision; see section 3.3.
MAX_SENTENCES = 16              # pooled sentences; the encoder separately truncates at MAX_TOKEN_LENGTH
EVIDENCE_STRENGTH = 1.0         # paired ablation: rerun with 0.0; only this loss coefficient changes
ATTENTION_SIZE = 256            # hidden size of the aspect-level fusion gate

# Existing DrugReview settings; unchanged from 03_/04_.
N_ITERATIONS = 10
MAX_TOKEN_LENGTH = 200
BATCH_SIZE = 128
PATIENCE = 3
NUM_WORKERS = 2
AUXILIARY_STRENGTH = 1.0
BACKBONES = ['albert', 'biobert']

AUX_CONFIG = {
    "folder": EXPERIMENT_FOLDER, "target": TRAINING_TARGET,
    "aspects": ASPECT_NAMES, "augmentation": USE_METADATA,
    "aux_strength": AUXILIARY_STRENGTH, "n_trials": N_ITERATIONS,
    "seed": RUN_SEED, "max_length": MAX_TOKEN_LENGTH,
    "batch_size": BATCH_SIZE, "patience": PATIENCE,
    "num_workers": NUM_WORKERS, "backbones": BACKBONES,
    "max_sentences": MAX_SENTENCES, "evidence_strength": EVIDENCE_STRENGTH, "attention_size": ATTENTION_SIZE,
    "alignment_version": "selected_quotes_visible_spans_v2",
}

EVIDENCE_CLASSES = ("NONE", "FAVOURABLE", "UNFAVOURABLE")
EVIDENCE_IGNORE = -100
EVIDENCE_ALIGNMENT_VERSION = AUX_CONFIG["alignment_version"]
AUX_VERSION = "evidence_hierarchical_v2"

AUX_ASPECTS = ("efficacy", "safety", "burden", "cost")
# The immutable GPT release keeps all five source states. Only supervision is collapsed.
AUX_SOURCE_STATES = ("NEGATIVE", "POSITIVE", "MIXED", "UNMENTIONED", "UNCLEAR")
AUX_STATES = ("NEGATIVE", "NEUTRAL", "POSITIVE")
AUX_STATE_TO_CLASS = {
    "NEGATIVE": "NEGATIVE", "POSITIVE": "POSITIVE",
    "MIXED": "NEUTRAL", "UNMENTIONED": "NEUTRAL", "UNCLEAR": "NEUTRAL",
}
# NEUTRAL is the pooled human-rubric class; it need not express a neutral attitude.
AUX_OVERALL = ("VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE")
AUX_RELEASE = {
    "protocol_id": "v2_ef356fdf87",
    "label_file_md5": "c22f6c05633214014db47c692e045ea1",
    "split_fingerprint": "038e1d09a2818dfcf6a91dce49e35a22",
    "n_eligible": 102061,
}
AUX_BACKBONES = {"albert": "albert-base-v2", "biobert": "dmis-lab/biobert-base-cased-v1.2"}

# Implementation fingerprint: sha256 of code cells except this editable configuration
# cell (index 10), joined with a single newline. Configuration and model-contract
# constants are saved separately in experiment_identity.json. Configuration-only
# ablations need no manual fingerprint changes; implementation edits do.
LEGACY_AUX_RUNTIME_SHA256S = []
AUX_RUNTIME_SHA256 = "33200b5af247e4b6492b463b3a1a0ca73b6212afda9cd012f54684dfcab16f7c"
OUTPUT_PATH = PROJECT_ROOT / EXPERIMENT_FOLDER / "outputs" / aux_output_version(AUX_CONFIG)
ALBERT_MODEL_NAME = AUX_BACKBONES["albert"]
BIOBERT_MODEL_NAME = AUX_BACKBONES["biobert"]
print("Experiment:", EXPERIMENT_FOLDER, "| max sentences:", MAX_SENTENCES, "| evidence strength:", EVIDENCE_STRENGTH, "| trials per model:", N_ITERATIONS)
print("Outputs:", OUTPUT_PATH)


Experiment: 07_Evidence_hierarchical | max sentences: 16 | evidence strength: 1.0 | trials per model: 10
Outputs: /content/drive/MyDrive/NLP_Projects/03_DrugReview/07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1


## 1.4 Mount drive & Load the data

In [8]:
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

def load_label_release(project_root, cfg):
    aux_validate_config(cfg)
    project_root = Path(project_root)
    source = project_root / "00_labelling/output_v2/v2_ef356fdf87/drugscom_plus_ai_labels_v2_ef356fdf87.csv"
    split_path = project_root / "00_split/data/DrugReview_split.csv"
    if aux_hash(source, "md5") != AUX_RELEASE["label_file_md5"]:
        raise ValueError("Label release changed. Restore the pinned release; do not silently relabel.")
    df = pd.read_csv(source, index_col=0).reset_index()
    split = pd.read_csv(split_path)
    required = {"uniqueID", "review", "eligible_both_streams", "rating", "ai_sent5_hard"}
    required |= {f"ai_{a}_state" for a in AUX_ASPECTS}
    required |= {f"ai_{a}_evidence" for a in AUX_ASPECTS}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing release columns: {required - set(df.columns)}")
    if not {"uniqueID", "split", "narrative_group", "is_audit_group"}.issubset(split.columns):
        raise ValueError("Incomplete split manifest.")
    for frame in (df, split):
        if frame.uniqueID.isna().any() or not frame.uniqueID.is_unique:
            raise ValueError("Missing or duplicate review IDs.")
    if set(df.uniqueID) != set(split.uniqueID):
        raise ValueError("Release and manifest ID sets differ.")
    fingerprint = hashlib.md5("|".join(
        f"{u}:{s}" for u, s in sorted(zip(split.uniqueID.astype(str), split["split"]))
    ).encode()).hexdigest()
    if fingerprint != AUX_RELEASE["split_fingerprint"]:
        raise ValueError("Frozen split changed.")
    if not split["split"].isin(["train", "val", "test", "audit"]).all():
        raise ValueError("Unknown split assignment.")
    if split.narrative_group.isna().any() or (split.groupby("narrative_group")["split"].nunique() > 1).any():
        raise ValueError("Missing narrative groups or narratives cross splits.")
    if not split.is_audit_group.isin([True, False]).all() or not (split.is_audit_group == split["split"].eq("audit")).all():
        raise ValueError("Audit reservation is inconsistent.")
    if not df.eligible_both_streams.isin([True, False]).all():
        raise ValueError("Invalid cohort eligibility flags.")
    df = df.loc[df.eligible_both_streams].copy()
    if len(df) != AUX_RELEASE["n_eligible"]:
        raise ValueError("Eligible cohort changed.")
    df = df.drop(columns=[c for c in split if c != "uniqueID" and c in df])
    df = df.merge(split, on="uniqueID", validate="one_to_one").sort_values("uniqueID").reset_index(drop=True)
    return df, split, source, split_path

df, split, source, split_path = load_label_release(PROJECT_ROOT, AUX_CONFIG)
print(f"Loaded {len(df):,} eligible reviews from {source}")


Mounted at /content/drive
Loaded 102,061 eligible reviews from /content/drive/MyDrive/NLP_Projects/03_DrugReview/00_labelling/output_v2/v2_ef356fdf87/drugscom_plus_ai_labels_v2_ef356fdf87.csv


In [9]:
# The loader has checked review IDs, the frozen label file and the saved split.
print("Saved partitions:", df["split"].value_counts().to_dict())


Saved partitions: {'train': 60950, 'test': 20298, 'val': 20256, 'audit': 557}


In [10]:
# Keep the existing review order and split; do not create a new random split.
print("Review IDs are sorted; training seeds do not change partition membership.")


Review IDs are sorted; training seeds do not change partition membership.


# 2. Data Exploration \& Master Data Preparation

## 2.1 Master Data Preparation

### 2.1.1 Text cleaning, sentence split and evidence alignment


In [11]:
def aux_clean_text(text):
    # Exactly the existing baseline cleaner, applied to the entire review.
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"#(\w+)", r" hashtag_\1 ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?;])\s+|[\r\n]+")

def aux_sentence_document(raw):
    """Baseline model text plus sentence spans in cleaned text and original review.
    Splitting never reconstructs or changes the encoder input."""
    raw = "" if pd.isna(raw) else str(raw)
    text = aux_clean_text(raw)
    cuts, last = [], 0
    for match in SENTENCE_BOUNDARY.finditer(raw):
        cuts.append((last, match.start()))
        last = match.end()
    cuts.append((last, len(raw)))
    sentences = []
    for raw_start, raw_end in cuts:
        start = len(aux_clean_text(raw[:raw_start]))
        end = len(aux_clean_text(raw[:raw_end]))
        while start < end and text[start].isspace():
            start += 1
        if end > start:
            sentences.append({"start": start, "end": end, "raw_start": raw_start,
                              "raw_end": raw_end, "text": text[start:end]})
    return {"text": text, "sentences": sentences}

def aux_align_evidence(document, evidence_json, source_state):
    """Align each normalized quote independently over the FULL review.
    Missing quotes remain unresolved; they do not invalidate other located quotes.
    Labels are weak selected-quote supervision, never exhaustive sentence truth."""
    diag = {"quotes_total": 0, "quotes_matched": 0, "quotes_unmatched": 0,
            "quotes_invalid": 0, "quotes_ambiguous_occurrence": 0,
            "quotes_unknown_direction": 0, "conflicting_sentences": 0,
            "invalid_payload": 0, "state_quote_conflict": 0}
    try:
        quotes = json.loads(evidence_json) if isinstance(evidence_json, str) else evidence_json
        if not isinstance(quotes, list):
            raise ValueError("Evidence must be a list.")
    except (ValueError, TypeError):
        quotes, diag["invalid_payload"] = [], 1
    known_none = source_state == "UNMENTIONED" and not quotes and not diag["invalid_payload"]
    if source_state == "UNMENTIONED" and quotes:
        diag["state_quote_conflict"] = 1
    records, directions = [], [set() for _ in document["sentences"]]
    for item in quotes:
        diag["quotes_total"] += 1
        if not isinstance(item, (list, tuple)) or len(item) != 2 or not isinstance(item[0], str):
            diag["quotes_invalid"] += 1
            continue
        quote, direction = item
        cleaned = aux_clean_text(quote)
        if not cleaned:
            diag["quotes_invalid"] += 1
            continue
        value = {"FAVOURABLE": 1, "UNFAVOURABLE": 2}.get(direction, EVIDENCE_IGNORE)
        if value == EVIDENCE_IGNORE:
            diag["quotes_unknown_direction"] += 1
        occurrences, start = [], 0
        while True:
            start = document["text"].find(cleaned, start)
            if start < 0:
                break
            occurrences.append((start, start + len(cleaned)))
            start += 1
        if not occurrences:
            diag["quotes_unmatched"] += 1
            continue
        # Repeated matching text does not reveal which occurrence was selected.
        # Keep this ambiguity explicit, rather than inventing sentence labels.
        if len(occurrences) != 1:
            diag["quotes_ambiguous_occurrence"] += 1
            continue
        span = occurrences[0]
        hits = [i for i, sentence in enumerate(document["sentences"])
                if span[0] < sentence["end"] and span[1] > sentence["start"]]
        if not hits:
            diag["quotes_unmatched"] += 1
            continue
        diag["quotes_matched"] += 1
        records.append({"span": list(span), "sentences": hits, "label": value})
        for i in hits:
            directions[i].add(value)
    labels = [0 if known_none else EVIDENCE_IGNORE for _ in directions]
    for i, values in enumerate(directions):
        if len(values) == 1 and EVIDENCE_IGNORE not in values and not diag["state_quote_conflict"]:
            labels[i] = next(iter(values))
        elif values:
            diag["conflicting_sentences"] += int(len(values) > 1)
    return {"labels": labels, "quotes": records, "known_none": known_none,
            "diagnostics": diag, "state_quote_conflict": bool(diag["state_quote_conflict"])}

def aux_alignment_digest(documents, alignments):
    digest = hashlib.sha256()
    for document, alignment in zip(documents, alignments):
        digest.update(json.dumps([document, alignment], sort_keys=True, separators=(",", ":")).encode())
        digest.update(b"\n")
    return digest.hexdigest()

if AUX_CONFIG["augmentation"]:
    raise ValueError("08_ is text only; metadata augmentation is not used here.")
df["sentence_document"] = df.review.map(aux_sentence_document)
df["model_text"] = df.sentence_document.map(lambda d: d["text"])
df["sentences"] = df.sentence_document.map(lambda d: [s["text"] for s in d["sentences"]])
if (df.sentences.map(len) == 0).any():
    raise ValueError("A review produced no sentence.")
evidence = np.full((len(df), MAX_SENTENCES, len(AUX_ASPECTS)), EVIDENCE_IGNORE, dtype=np.int64)
coverage = {a: {} for a in AUX_ASPECTS}
alignments = []
for i, row in enumerate(df.itertuples(index=False)):
    aligned = []
    for k, aspect in enumerate(AUX_ASPECTS):
        item = aux_align_evidence(row.sentence_document, getattr(row, f"ai_{aspect}_evidence"),
                                  getattr(row, f"ai_{aspect}_state"))
        labels = item["labels"][:MAX_SENTENCES]
        evidence[i, :len(labels), k] = labels
        aligned.append(item)
        for key, value in item["diagnostics"].items():
            coverage[aspect][key] = coverage[aspect].get(key, 0) + value
    alignments.append(aligned)
df["evidence_alignment"] = alignments
df["evidence_labels"] = list(evidence)  # provisional; tokenizer visibility is applied before training
print("Whole-review input uses the unchanged baseline cleaner.")
print("Source alignment diagnostics (before sentence/token limits):", coverage)
print("Reviews above sentence pooling limit:", int((df.sentences.map(len) > MAX_SENTENCES).sum()))


Whole-review input uses the unchanged baseline cleaner.
Source alignment diagnostics (before sentence/token limits): {'efficacy': {'quotes_total': 196344, 'quotes_matched': 190799, 'quotes_unmatched': 5365, 'quotes_invalid': 0, 'quotes_ambiguous_occurrence': 180, 'quotes_unknown_direction': 4908, 'conflicting_sentences': 1369, 'invalid_payload': 0, 'state_quote_conflict': 510}, 'safety': {'quotes_total': 122856, 'quotes_matched': 118195, 'quotes_unmatched': 4599, 'quotes_invalid': 0, 'quotes_ambiguous_occurrence': 62, 'quotes_unknown_direction': 1850, 'conflicting_sentences': 770, 'invalid_payload': 0, 'state_quote_conflict': 57}, 'burden': {'quotes_total': 84694, 'quotes_matched': 81902, 'quotes_unmatched': 2758, 'quotes_invalid': 0, 'quotes_ambiguous_occurrence': 34, 'quotes_unknown_direction': 4043, 'conflicting_sentences': 317, 'invalid_payload': 0, 'state_quote_conflict': 1701}, 'cost': {'quotes_total': 9363, 'quotes_matched': 9110, 'quotes_unmatched': 252, 'quotes_invalid': 0, 'q

### 2.1.2 Validate & encode aspect labels

Aspect targets: NEGATIVE = 0, NEUTRAL = 1, POSITIVE = 2. MIXED, UNMENTIONED and UNCLEAR collapse to NEUTRAL for the review-level aspect task; source states are preserved. The holistic target has five classes.

Evidence targets are NONE = 0, FAVOURABLE = 1, UNFAVOURABLE = 2, UNKNOWN = −100. NONE is supervised only for an explicitly UNMENTIONED aspect with an empty valid quote list. In reviews with selected quotes, other sentences remain UNKNOWN. Empty UNCLEAR, malformed quotes, unmatched or ambiguous occurrences, and conflicting sentence directions are not negative evidence. Known targets are further restricted to completely visible quotations after tokenization; unlocated quotes remain unresolved.


In [12]:
# Validate all targets, so baseline and joint conditions have the same cohort.
if not df.ai_sent5_hard.isin(AUX_OVERALL).all():
    raise ValueError("Invalid holistic label; never replace failed labels with neutral.")
rating = pd.to_numeric(df.rating, errors="coerce").round()
if not rating.between(1, 10).all():
    raise ValueError("Invalid rating would change the common cohort.")
df["original_id"] = ((rating.astype(int) - 1) // 2).astype(int)
df["holistic_id"] = df.ai_sent5_hard.map({x: i for i, x in enumerate(AUX_OVERALL)}).astype(int)
for aspect in AUX_ASPECTS:
    col = f"ai_{aspect}_state"
    if not df[col].isin(AUX_SOURCE_STATES).all():
        raise ValueError(f"Invalid five-state label: {col}")
    df[f"{aspect}_id"] = df[col].map(AUX_STATE_TO_CLASS).map(
        {x: i for i, x in enumerate(AUX_STATES)}).astype(int)

for aspect in ASPECT_NAMES:
    print(aspect, df[f"{aspect}_id"].value_counts().sort_index().to_dict())

if not np.isin(np.unique(evidence), [EVIDENCE_IGNORE, 0, 1, 2]).all():
    raise ValueError("Invalid sentence-level evidence label.")


efficacy {0: 19761, 1: 17551, 2: 64749}
safety {0: 51921, 1: 34895, 2: 15245}
burden {0: 40844, 1: 51341, 2: 9876}
cost {0: 4689, 1: 94905, 2: 2467}


## 2.3 Master Data Split

Use the existing train, validation, test and reserved audit assignments from `00_split`. Training and parameter selection use only train and validation rows.


In [13]:
frames = {s: df.loc[df["split"].eq(s)].copy().reset_index(drop=True) for s in ("train", "val", "test", "audit")}
if any(len(frame) == 0 for frame in frames.values()):
    raise ValueError("Each frozen partition must be nonempty.")
identity = {
    "release": AUX_RELEASE, "split_file_sha256": aux_hash(split_path),
    "cohort_sha256": hashlib.sha256("|".join(df.uniqueID.astype(str)).encode()).hexdigest(),
    "split_counts": {s: len(f) for s, f in frames.items()},
    "text_sha256": hashlib.sha256("\n".join(df.model_text).encode()).hexdigest(),
    "evidence_sha256": hashlib.sha256(evidence.tobytes()).hexdigest(),
    "evidence_coverage": coverage,
    "alignment_version": EVIDENCE_ALIGNMENT_VERSION,
    "alignment_sha256": aux_alignment_digest(df.sentence_document, df.evidence_alignment),
    "label_file_sha256": aux_hash(source),
}

print("Partitions:", identity["split_counts"])


Partitions: {'train': 60950, 'val': 20256, 'test': 20298, 'audit': 557}


# 3. ALBERT \& BioBERT Models (Evidence-supervised hierarchical)


## 3.1 Shared helper functions

Losses retain the existing overall and aspect targets. Evidence classification uses only known sentence targets; class weights are computed separately for each tokenizer from visible TRAIN targets. Localization reports top-attention selected-quote hit rate and attention mass on sentences containing fully visible, unambiguous selected quotes. Its denominator is reviews with at least one such quote. Unselected sentences are not assumed negative; these are annotation-localization measures, not exhaustive precision, recall, or human validity.

Completed outputs are verified against data, alignment, code and configuration identities. The evidence-strength 0 and 1 runs use separate folders and the same model/trial initialization and selection rule.


In [14]:
def aux_task_labels(frame, cfg):
    labels = {a: frame[f"{a}_id"].to_numpy() for a in cfg["aspects"]}
    if cfg["target"] != "aspects":
        labels["overall"] = frame[f"{cfg['target']}_id"].to_numpy()
    labels["evidence"] = np.stack(frame["evidence_labels"].to_numpy())          # (N, S, K)
    return labels

def aux_criteria(train_labels, device):
    # Retain balanced CE within each task, calculated on TRAIN only.
    criteria, weights = {}, {}
    for task, values in train_labels.items():
        values = np.asarray(values)
        if task == "evidence":
            if values.ndim != 3 or values.shape[1:] != (MAX_SENTENCES, len(AUX_ASPECTS)):
                raise ValueError("Invalid sentence-level evidence targets.")
            criteria[task], weights[task] = [], {}
            for k, aspect in enumerate(AUX_ASPECTS):
                labelled = values[:, :, k][values[:, :, k] != EVIDENCE_IGNORE]
                counts = np.bincount(labelled, minlength=len(EVIDENCE_CLASSES))
                seen = counts > 0
                weight = np.zeros(len(EVIDENCE_CLASSES), dtype=np.float32)
                weight[seen] = len(labelled) / (seen.sum() * counts[seen])
                weights[task][aspect] = weight.tolist()
                criteria[task].append(nn.CrossEntropyLoss(weight=torch.tensor(weight, device=device), ignore_index=EVIDENCE_IGNORE))
            continue
        n_classes = len(AUX_OVERALL) if task == "overall" else len(AUX_STATES)
        if values.ndim != 1 or len(values) == 0 or not np.issubdtype(values.dtype, np.integer) or not ((values >= 0) & (values < n_classes)).all():
            raise ValueError(f"Invalid {n_classes}-class training targets: {task}")
        counts = np.bincount(values, minlength=n_classes)
        seen = counts > 0
        weight = np.zeros(n_classes, dtype=np.float32)
        weight[seen] = len(values) / (seen.sum() * counts[seen])
        weights[task] = weight.tolist()
        criteria[task] = nn.CrossEntropyLoss(weight=torch.tensor(weight, device=device))
    return criteria, weights

def aux_loss(logits, targets, criteria, strength, evidence_strength=0.0):
    terms = {k: criteria[k](logits[k].float(), targets[k]) for k in logits if not k.startswith("evidence")}
    aspect_terms = [v for k, v in terms.items() if k != "overall"]
    total = terms["overall"] if "overall" in terms else torch.stack(aspect_terms).mean()
    if "overall" in terms and aspect_terms:
        total = total + strength * torch.stack(aspect_terms).mean()
    if "evidence" in logits:
        evidence_logits = logits["evidence"].float()                                   # (B, S, K, 3)
        present = logits["evidence_present"]                                           # (B, S)
        evidence_targets = targets["evidence"].masked_fill(~present.unsqueeze(-1), EVIDENCE_IGNORE)
        evidence_terms = []
        for k in range(evidence_logits.shape[2]):
            if (evidence_targets[:, :, k] != EVIDENCE_IGNORE).any():
                evidence_terms.append(criteria["evidence"][k](
                    evidence_logits[:, :, k, :].reshape(-1, len(EVIDENCE_CLASSES)), evidence_targets[:, :, k].reshape(-1)))
        if evidence_terms:
            terms["evidence"] = torch.stack(evidence_terms).mean()
            total = total + evidence_strength * terms["evidence"]
    return total, terms

def aux_metrics(y, pred, ordinal=False, n_classes=None):
    # Aspect macro F1 averages three classes; overall metrics retain five.
    class_ids = range(n_classes if n_classes else (len(AUX_OVERALL) if ordinal else len(AUX_STATES)))
    out = {
        "n": len(y), "accuracy": float(accuracy_score(y, pred)),
        "weighted_f1": float(f1_score(y, pred, labels=class_ids, average="weighted", zero_division=0)),
        "macro_f1": float(f1_score(y, pred, labels=class_ids, average="macro", zero_division=0)),
    }
    if ordinal:
        qwk = float(cohen_kappa_score(y, pred, labels=range(5), weights="quadratic"))
        out.update(mae=float(np.mean(np.abs(np.asarray(y) - pred))), qwk=qwk if np.isfinite(qwk) else None)
    return out

def aux_predict(model, loader, device):
    """Predict every review-level task; collect fusion weights, sentence attention and sentence-level evidence predictions."""
    model.eval()
    probs, labels, extras = {}, {}, {}
    with torch.no_grad():
        for batch in loader:
            logits, batch_extras = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                                         batch["sentence_ids"].to(device), return_extras=True)
            for task, values in logits.items():
                if task.startswith("evidence"):
                    continue
                require_finite_tensor(values, f"{task} prediction logits")
                probabilities = values.float().softmax(-1).cpu().numpy()
                require_probabilities(probabilities, values.shape[-1])
                probs.setdefault(task, []).append(probabilities)
                labels.setdefault(task, []).append(batch["targets"][task].numpy())
            present = logits["evidence_present"].cpu()
            evidence_true = batch["targets"]["evidence"].masked_fill(~present.unsqueeze(-1), EVIDENCE_IGNORE)
            extras.setdefault("evidence_true", []).append(evidence_true.numpy())
            for key, value in batch_extras.items():
                if value is None:
                    continue
                require_finite_tensor(value.float(), key)
                extras.setdefault(key, []).append(value.cpu().numpy())
    extras = {k: np.concatenate(v) for k, v in extras.items()}
    return {k: np.concatenate(v) for k, v in probs.items()}, {k: np.concatenate(v) for k, v in labels.items()}, extras

def aux_selection(probs, labels):
    for task, p in probs.items():
        require_probabilities(p, 5 if task == "overall" else 3)
    if "overall" in probs:
        return float(f1_score(labels["overall"], probs["overall"].argmax(1), average="weighted", zero_division=0))
    return float(np.mean([f1_score(labels[k], p.argmax(1), average="weighted", zero_division=0) for k, p in probs.items()]))

def aux_trial_plan(cfg):
    rng = np.random.RandomState(cfg["seed"])
    return [{"lr": float(10**rng.uniform(np.log10(1e-5), np.log10(3e-5))),
             "epochs": int(rng.randint(4, 10)), "dropout": float(rng.uniform(0.0, 0.25)),
             "seed": cfg["seed"] + i} for i in range(cfg["n_trials"])]

def aux_evidence_metrics(evidence_true, evidence_pred, attention, quote_mask):
    """Classification on known targets is separate from selected-quote localization.
    Unknown sentences never become negative targets. Quote localization is not
    exhaustive evidence precision/recall and may penalize valid unselected text."""
    y_all, pred_all = np.asarray(evidence_true), np.asarray(evidence_pred)
    attention, quote_mask = np.asarray(attention), np.asarray(quote_mask, dtype=bool)
    if y_all.shape != pred_all.shape or attention.shape != y_all.shape or quote_mask.shape != y_all.shape:
        raise ValueError("Evidence/localization shapes differ.")
    if not np.isfinite(attention).all() or (attention < 0).any() or not np.allclose(attention.sum(1), 1, atol=1e-5):
        raise ValueError("Invalid sentence attention.")
    known = y_all != EVIDENCE_IGNORE
    y, pred = y_all[known], pred_all[known]
    out = {"accuracy": None, "weighted_f1": None, "macro_f1": None,
           "n": int(known.sum()), "known_positive_n": int((y > 0).sum()),
           "known_negative_n": int((y == 0).sum()), "unknown_sentence_slots_n": int((~known).sum())}
    if len(y):
        out.update(aux_metrics(y, pred, False, n_classes=len(EVIDENCE_CLASSES)))
        out["known_target_mention_f1"] = float(f1_score(y > 0, pred > 0, zero_division=0))
        out["known_target_mention_precision"] = float(((pred > 0) & (y > 0)).sum() / max(int((pred > 0).sum()), 1))
        out["known_target_mention_recall"] = float(((pred > 0) & (y > 0)).sum() / max(int((y > 0).sum()), 1))
        positive = y > 0
        out["selected_positive_direction_accuracy"] = float((y[positive] == pred[positive]).mean()) if positive.any() else None
    eligible = quote_mask.any(1)
    top = attention.argmax(1)
    out["localization_reviews_n"] = int(eligible.sum())
    out["localization_excluded_reviews_n"] = int((~eligible).sum())
    out["top1_selected_quote_hit"] = float(quote_mask[np.arange(len(top)), top][eligible].mean()) if eligible.any() else None
    mass = (attention * quote_mask).sum(1)
    out["selected_quote_attention_mass"] = float(mass[eligible].mean()) if eligible.any() else None
    return out

def aux_export(model, loader, frame, cfg, model_name, split_name, output, device):
    probs, labels, extras = aux_predict(model, loader, device)
    audit_columns = [c for c in ("is_human_annotated", "is_audit_group", "audit_stratum") if c in frame]
    result = frame[["uniqueID", "narrative_group", "split"] + audit_columns].copy()
    metrics = []
    for task, p in probs.items():
        names = AUX_OVERALL if task == "overall" else AUX_STATES
        if p.ndim != 2 or p.shape != (len(frame), len(names)):
            raise ValueError(f"Unexpected probability shape for {task}: {p.shape}")
        require_probabilities(p, len(names))
        pred, y = p.argmax(1), labels[task]
        result[f"{task}_true_id"] = y
        result[f"{task}_pred_id"] = pred
        result[f"{task}_pred"] = np.asarray(names)[pred]
        for k, name in enumerate(names):
            result[f"{task}_p_{name}"] = p[:, k]
        row = {"model": model_name, "variant": "evidence_hierarchical", "task": task, "partition": split_name, "reference_source": "rating" if cfg["target"] == "original" and task == "overall" else "GPT-4o-mini", "human_agreement": False, **aux_metrics(y, pred, task == "overall")}
        metrics.append(row)
        if split_name == "test":
            pd.DataFrame(classification_report(y, pred, labels=range(len(names)), target_names=names,
                output_dict=True, zero_division=0)).T.to_csv(output / f"{model_name}_{task}_class_report.csv")
            pd.DataFrame(confusion_matrix(y, pred, labels=range(len(names))), index=names, columns=names).to_csv(
                output / f"{model_name}_{task}_confusion.csv")
        if task != "overall":
            # Native three-class predictions; these compatibility columns are exact aliases.
            result[f"{task}_true_3class_id"] = y
            result[f"{task}_pred_3class_id"] = pred
            source_col = f"ai_{task}_state"
            if source_col in frame:
                result[f"{task}_source_state"] = frame[source_col].to_numpy()
                result[f"{task}_is_mentioned"] = frame[source_col].ne("UNMENTIONED").to_numpy()
    # Classification and selected-quote localization have different denominators.
    evidence_true, evidence_pred = extras["evidence_true"], extras["evidence_pred"].astype(np.int64)
    metadata = loader.dataset.evidence_metadata
    if len(metadata) != len(frame):
        raise ValueError("Missing tokenizer-visible annotation metadata.")
    quote_masks = np.asarray([item["quote_mask"] for item in metadata], dtype=bool)
    coverage_rows = []
    for k, aspect in enumerate(cfg["aspects"]):
        measure = aux_evidence_metrics(evidence_true[:, :, k], evidence_pred[:, :, k],
                                       extras["sentence_attention"][:, k, :], quote_masks[:, :, k])
        totals = {key: int(sum(item["diagnostics"][k][key] for item in metadata))
                  for key in metadata[0]["diagnostics"][k]}
        coverage_rows.append({"aspect": aspect, "reviews_n": len(frame), **totals})
        row = {"model": model_name, "variant": "evidence_hierarchical", "task": f"evidence_{aspect}",
               "partition": split_name, "reference_source": "GPT-4o-mini selected nonexhaustive quotes",
               "human_agreement": False, "evidence_strength": cfg["evidence_strength"], **measure}
        metrics.append(row)
        mask = evidence_true[:, :, k] != EVIDENCE_IGNORE
        if split_name == "test" and mask.any():
            pd.DataFrame(confusion_matrix(evidence_true[:, :, k][mask], evidence_pred[:, :, k][mask],
                         labels=range(len(EVIDENCE_CLASSES))), index=EVIDENCE_CLASSES,
                         columns=EVIDENCE_CLASSES).to_csv(output / f"{model_name}_evidence_{aspect}_confusion.csv")
    pd.DataFrame(coverage_rows).to_csv(output / f"{model_name}_{split_name}_evidence_coverage.csv", index=False)
    # Review-specific aspect fusion weights.
    fusion = require_probabilities(extras["fusion"], len(cfg["aspects"]))
    if fusion.shape[0] != len(frame):
        raise ValueError(f"Unexpected fusion shape: {fusion.shape}")
    for k, aspect in enumerate(cfg["aspects"]):
        result[f"fusion_w_{aspect}"] = fusion[:, k]
    if split_name == "test":
        summary = result.groupby("overall_true_id")[[f"fusion_w_{a}" for a in cfg["aspects"]]].mean()
        summary.index = np.asarray(AUX_OVERALL)[summary.index.to_numpy()]
        summary.to_csv(output / f"{model_name}_fusion_by_holistic_class.csv")
    # Export only visible text/spans. Original full sentence boundaries are metadata,
    # never displayed as if the encoder saw truncated content.
    attention = extras["sentence_attention"]
    for k, aspect in enumerate(cfg["aspects"]):
        top = attention[:, k, :].argmax(1)
        selected = [metadata[i]["sentences"][int(index)] for i, index in enumerate(top)]
        result[f"{aspect}_top_visible_sentence"] = [item["visible_text"] for item in selected]
        result[f"{aspect}_top_sentence_weight"] = attention[np.arange(len(frame)), k, top]
        result[f"{aspect}_top_sentence_visible_start"] = [item["visible_start"] for item in selected]
        result[f"{aspect}_top_sentence_visible_end"] = [item["visible_end"] for item in selected]
        result[f"{aspect}_top_sentence_partially_truncated"] = [item["partially_truncated"] for item in selected]
        result[f"{aspect}_has_visible_selected_quote"] = quote_masks[:, :, k].any(1)
        result[f"{aspect}_top1_selected_quote_hit"] = [bool(quote_masks[i, int(s), k]) if quote_masks[i, :, k].any() else None
                                                            for i, s in enumerate(top)]
    if split_name == "audit":
        with open(output / f"{model_name}_audit_visible_sentences.jsonl", "w", encoding="utf-8") as stream:
            for i, item in enumerate(metadata):
                stream.write(json.dumps({"uniqueID": int(frame.iloc[i].uniqueID), **item,
                    "sentence_attention": attention[i].tolist()}, allow_nan=False) + "\n")
    result.to_csv(output / f"{model_name}_{split_name}_predictions.csv", index=False)
    return metrics

def prepare_experiment(project_root, cfg, frames, identity):
    """Check saved settings and prepare the output folder and training losses."""
    import transformers
    import sklearn
    output = Path(project_root) / cfg["folder"] / "outputs" / aux_output_version(cfg)
    output.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    plan = aux_trial_plan(cfg)
    comparable_cfg = {k: v for k, v in cfg.items() if k not in ("folder", "backbones", "notebook")}
    signature = {"version": AUX_VERSION, "runtime_sha256": AUX_RUNTIME_SHA256, "model_contract": aux_model_contract(),
        "config": comparable_cfg, "data": identity, "trials": plan,
        "aspect_schema": {"classes": list(AUX_STATES), "source_mapping": AUX_STATE_TO_CLASS},
        "evidence_schema": {"classes": list(EVIDENCE_CLASSES), "ignore": EVIDENCE_IGNORE, "max_sentences": cfg["max_sentences"],
                            "alignment_version": EVIDENCE_ALIGNMENT_VERSION, "unquoted": "UNKNOWN",
                            "known_none": "UNMENTIONED with empty valid evidence list", "visibility": "full quoted span retained"},
        "python": platform.python_version(), "torch": torch.__version__,
        "transformers": transformers.__version__, "sklearn": sklearn.__version__}
    # Never silently load stale checkpoints when data, settings, code or libraries differ.
    identity_path = output / "experiment_identity.json"
    if identity_path.exists():
        saved_signature = json.loads(identity_path.read_text())
        expected_signature = dict(signature)
        # Accept the previous cell layout only when every data/training setting matches.
        # Keep the original saved identity; do not relabel an old run as a new one.
        if saved_signature.get("runtime_sha256") in LEGACY_AUX_RUNTIME_SHA256S:
            if not all((output / model_name / "complete.json").exists() for model_name in cfg["backbones"]):
                raise ValueError("This unfinished run uses earlier training code. Keep its original notebook to resume it; do not mix implementations in one run.")
            expected_signature["runtime_sha256"] = saved_signature["runtime_sha256"]
        if saved_signature != expected_signature:
            raise ValueError("Existing outputs belong to a different configuration. Use a new output version/folder.")
        signature = saved_signature
    aux_json(identity_path, signature)
    aux_json(output / "run_environment.json", {"device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "notebook_code_sha256": AUX_RUNTIME_SHA256})
    for name in ("train", "val", "test", "audit"):
        frames[name][["uniqueID", "split", "narrative_group"]].to_csv(output / f"{name}_ids.csv", index=False)
    print("Partitions:", identity["split_counts"], "| target:", cfg["target"], "| aspects:", cfg["aspects"], "| evidence strength:", cfg["evidence_strength"])
    train_labels = aux_task_labels(frames["train"], cfg)
    # Review-level weights are tokenizer-independent. Evidence weights are computed
    # after each backbone tokenizes TRAIN; no invisible targets influence them.
    train_labels.pop("evidence")
    criteria, class_weights = aux_criteria(train_labels, device)
    aux_json(output / "review_train_class_weights.json", class_weights)
    return output, device, plan, criteria, class_weights


## 3.2 Data splits \& labels

In [15]:
train_labels = aux_task_labels(frames["train"], AUX_CONFIG)
val_labels = aux_task_labels(frames["val"], AUX_CONFIG)
test_labels = aux_task_labels(frames["test"], AUX_CONFIG)

output, device, plan, criteria, class_weights = prepare_experiment(
    PROJECT_ROOT, AUX_CONFIG, frames, identity
)
print("Parameter search:")
print(pd.DataFrame(plan).to_string(index=False))
print("Class weights from training data:", class_weights)


Partitions: {'train': 60950, 'val': 20256, 'test': 20298, 'audit': 557} | target: holistic | aspects: ['efficacy', 'safety', 'burden', 'cost'] | evidence strength: 1.0
Parameter search:
      lr  epochs  dropout  seed
0.000012       6 0.085351  2025
0.000011       7 0.097059  2026
0.000013       9 0.195692  2027
0.000011       9 0.200246  2028
0.000016       4 0.094601  2029
0.000018       5 0.000793  2030
0.000014       9 0.183296  2031
0.000011       7 0.062150  2032
0.000021       6 0.155208  2033
0.000030       9 0.236536  2034
Class weights from training data: {'efficacy': [1.7274608612060547, 1.9261155128479004, 0.5257800221443176], 'safety': [0.653416097164154, 0.9769037365913391, 2.242457628250122], 'burden': [0.8348743319511414, 0.6606616377830505, 3.4652340412139893], 'cost': [7.353118419647217, 0.3579587936401367, 14.207459449768066], 'overall': [1.098297119140625, 1.206214189529419, 6.200407028198242, 0.6612063646316528, 0.6302021145820618]}


## 3.3 Evidence-supervised hierarchical model components

Whole-review encoder → contextual sentence states → evidence-weighted aspect representations. Each aspect representation feeds its three-class aspect head **and**, in parallel, the fusion gate. The holistic head consumes `[CLS ; fused aspect representations]`.

* **Sentence pooling:** at most `MAX_SENTENCES` sentences are pooled from retained token offsets. The encoder only sees `MAX_TOKEN_LENGTH` tokens; later sentences inside that window can affect contextual states and `[CLS]` even if not pooled.
* **Weak evidence supervision:** unique full-review quote spans can cross sentence boundaries. Only fully visible quotes supervise intersecting sentences. Other selected quotes survive an unmatched quote; ambiguous occurrences/directions are recorded and ignored. Direction conflicts are resolved per sentence after visibility filtering: a clean sentence in a cross-sentence quote survives, and an invisible opposing quote does not suppress it. Unselected sentences remain UNKNOWN, apart from explicit empty-quote UNMENTIONED reviews.
* **Sentence attention:** a softmax over predicted `log(P(FAVOURABLE) + P(UNFAVOURABLE))` within visible sentences. The class head and resulting attention must be evaluated separately.
* **Paired ablation:** `EVIDENCE_STRENGTH = 0.0` uses exactly the same architecture, heads, seed, trial plan, initialization, data order, dropout RNG and validation overall weighted-F1 selection as `1.0`. Only the evidence-loss coefficient differs. Evidence heads can still receive task gradients through attention when direct evidence loss is zero.

Loss = overall weighted cross-entropy + `AUXILIARY_STRENGTH` × mean aspect loss + `EVIDENCE_STRENGTH` × mean known-target evidence loss. Weights use TRAIN only and evidence weights use each tokenizer's visible targets. Evidence classification F1 is **not** attention localization. Top-attention selected-quote hit rate and selected-quote attention mass are reported only among reviews with fully visible known quotes, alongside coverage/exclusions. Contextual states and the CLS residual prevent an exclusive-evidence interpretation.


In [16]:

def aux_verify_completed_run(model_output):
    """A completion shortcut requires a complete, unchanged set of run artifacts."""
    model_output = Path(model_output)
    complete = json.loads((model_output / "complete.json").read_text())
    artifacts = complete.get("artifacts")
    if not isinstance(artifacts, dict) or not artifacts:
        raise ValueError("Completed run has an empty or invalid artifact manifest.")
    model_name = model_output.name
    required = {"best_model.pth", "metrics.csv", "selected_parameters.json"}
    required.update(f"{model_name}_{part}_predictions.csv" for part in ("val", "test", "audit"))
    required.update(("tokenization_identity.json", "train_class_weights.json"))
    missing = required - set(artifacts)
    if missing:
        raise ValueError(f"Completed run manifest omits required artifacts: {sorted(missing)}")
    base = model_output.resolve()
    for filename, digest in artifacts.items():
        path = model_output / filename
        if Path(filename).is_absolute() or base not in path.resolve().parents:
            raise ValueError(f"Completed artifact escapes its run directory: {filename}")
        if not path.is_file() or aux_hash(path) != digest:
            raise ValueError(f"Completed artifact changed: {model_name}/{filename}")
    verify_completed_numerics(model_output)
    return complete

def verify_completed_numerics(model_output):
    for prediction_path in model_output.glob("*_predictions.csv"):
        saved = pd.read_csv(prediction_path)
        for task in ["overall", *AUX_ASPECTS]:
            names = AUX_OVERALL if task == "overall" else AUX_STATES
            columns = [f"{task}_p_{name}" for name in names]
            if all(c in saved for c in columns):
                require_probabilities(saved[columns].to_numpy(), len(names))
        fusion_columns = [c for c in saved.columns if c.startswith("fusion_w_")]
        if fusion_columns:
            require_probabilities(saved[fusion_columns].to_numpy(), len(fusion_columns))
    payload = torch.load(model_output / "best_model.pth", map_location="cpu", weights_only=True)
    for name, value in payload["state_dict"].items():
        if value.is_floating_point():
            require_finite_tensor(value, f"saved model state {name}")

class AuxiliaryDataset(Dataset):
    """Labels/annotation metadata never enter model.forward."""
    def __init__(self, encoded, labels, sentence_ids, evidence_metadata=None):
        self.encoded = encoded
        self.sentence_ids = sentence_ids
        self.labels = {k: torch.as_tensor(v, dtype=torch.long) for k, v in labels.items()}
        self.evidence_metadata = evidence_metadata or []

    def __len__(self):
        return len(self.encoded["input_ids"])

    def __getitem__(self, index):
        return {"input_ids": self.encoded["input_ids"][index],
                "attention_mask": self.encoded["attention_mask"][index],
                "sentence_ids": self.sentence_ids[index],
                "targets": {k: v[index] for k, v in self.labels.items()}}

def aux_sentence_ids(offsets, documents, max_sentences):
    """Assign a token only when its non-whitespace span belongs to one sentence."""
    ids = torch.full(offsets.shape[:2], -1, dtype=torch.long)
    for i, document in enumerate(documents):
        text = document["text"]
        for j, (start, end) in enumerate(offsets[i].tolist()):
            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1
            if end <= start:
                continue
            for k, sentence in enumerate(document["sentences"][:max_sentences]):
                if start >= sentence["start"] and end <= sentence["end"]:
                    ids[i, j] = k
                    break
    return ids

def aux_span_visible(text, span, intervals):
    """Every nonspace quote character must occur inside retained token offsets."""
    start, end = span
    covered = [False] * (end - start)
    for left, right in intervals:
        for index in range(max(start, left), min(end, right)):
            covered[index - start] = True
    return all(seen or text[start + i].isspace() for i, seen in enumerate(covered))

def aux_visible_evidence(document, alignments, offsets, sentence_ids, max_sentences):
    """Mask entire partially invisible quotations, retaining other visible quotes.
    Token and sentence-limit exclusions are reported independently."""
    labels = np.full((max_sentences, len(alignments)), EVIDENCE_IGNORE, dtype=np.int64)
    quote_mask = np.zeros_like(labels, dtype=bool)
    offsets = offsets.tolist() if hasattr(offsets, "tolist") else offsets
    sentence_ids = sentence_ids.tolist() if hasattr(sentence_ids, "tolist") else sentence_ids
    all_intervals = [(int(a), int(b)) for a, b in offsets if b > a]
    pooled_intervals = [(int(a), int(b)) for (a, b), sid in zip(offsets, sentence_ids) if b > a and sid >= 0]
    visible = []
    for i, sentence in enumerate(document["sentences"][:max_sentences]):
        intervals = [(max(a, sentence["start"]), min(b, sentence["end"]))
                     for (a, b), sid in zip(offsets, sentence_ids) if sid == i and b > a]
        start = min((a for a, b in intervals), default=sentence["start"])
        end = max((b for a, b in intervals), default=start)
        complete = bool(intervals) and aux_span_visible(document["text"], (sentence["start"], sentence["end"]), intervals)
        visible.append({"sentence_index": i, "cleaned_sentence_start": sentence["start"],
                        "cleaned_sentence_end": sentence["end"], "source_raw_start": sentence["raw_start"],
                        "source_raw_end": sentence["raw_end"], "visible_start": start, "visible_end": end,
                        "visible_text": document["text"][start:end], "token_spans": [list(x) for x in intervals],
                        "has_visible_tokens": bool(intervals), "partially_truncated": bool(intervals) and not complete,
                        "fully_unseen": not bool(intervals)})
    diagnostics = []
    for k, aligned in enumerate(alignments):
        diag = dict(aligned["diagnostics"])
        diag.update(quotes_cut_by_tokens=0, quotes_cut_by_sentence_limit=0, quotes_visible=0,
                    quotes_excluded_direction=0, quotes_excluded_conflict=0,
                    quotes_with_supervised_sentence=0, visible_conflicting_sentences=0,
                    positive_targets=0, known_negative_targets=0, unknown_targets=0)
        if aligned["known_none"]:
            for i, sentence in enumerate(visible):
                if sentence["has_visible_tokens"]:
                    labels[i, k] = 0
        # Rebuild direction sets from fully visible quotes. An unseen opponent
        # cannot create a conflict in the retained evidence. Resolve conflicts
        # per sentence, so a clean part of a cross-sentence quote survives.
        directions = [set() for _ in range(max_sentences)]
        visible_quotes = []
        for quote in aligned["quotes"]:
            token_visible = aux_span_visible(document["text"], quote["span"], all_intervals)
            within_limit = all(i < max_sentences for i in quote["sentences"])
            diag["quotes_cut_by_tokens"] += int(not token_visible)
            diag["quotes_cut_by_sentence_limit"] += int(not within_limit)
            if aligned["state_quote_conflict"]:
                diag["quotes_excluded_conflict"] += 1
                continue
            if not token_visible or not within_limit or not aux_span_visible(document["text"], quote["span"], pooled_intervals):
                continue
            if quote["label"] == EVIDENCE_IGNORE:
                diag["quotes_excluded_direction"] += 1
            else:
                diag["quotes_visible"] += 1
            visible_quotes.append(quote)
            for i in quote["sentences"]:
                directions[i].add(quote["label"])
        for i, values in enumerate(directions):
            if len(values) == 1 and EVIDENCE_IGNORE not in values:
                labels[i, k] = next(iter(values))
                quote_mask[i, k] = True
            elif len(values) > 1:
                diag["visible_conflicting_sentences"] += 1
        for quote in visible_quotes:
            if quote["label"] == EVIDENCE_IGNORE:
                continue
            survives = any(quote_mask[i, k] and labels[i, k] == quote["label"] for i in quote["sentences"])
            diag["quotes_with_supervised_sentence"] += int(survives)
            diag["quotes_excluded_conflict"] += int(not survives)
        present = np.array([s["has_visible_tokens"] for s in visible] + [False] * (max_sentences - len(visible)))
        diag["positive_targets"] = int((labels[:, k] > 0).sum())
        diag["known_negative_targets"] = int((labels[:, k] == 0).sum())
        diag["unknown_targets"] = int(((labels[:, k] == EVIDENCE_IGNORE) & present).sum())
        diag["reviews_with_visible_quote"] = int(quote_mask[:, k].any())
        diag["reviews_without_visible_quote"] = int(not quote_mask[:, k].any())
        diagnostics.append(diag)
    return labels, {"sentences": visible, "quote_mask": quote_mask.tolist(), "diagnostics": diagnostics}

def aux_encode(tokenizer, frame, cfg):
    if not getattr(tokenizer, "is_fast", False):
        raise ValueError("A fast tokenizer is required for character offsets.")
    encoded = tokenizer(frame.model_text.tolist(), truncation=True, padding="max_length",
                        max_length=cfg["max_length"], return_tensors="pt", return_offsets_mapping=True)
    offsets = encoded.pop("offset_mapping")
    sentence_ids = aux_sentence_ids(offsets, frame.sentence_document.tolist(), cfg["max_sentences"])
    if not ((sentence_ids >= 0).sum(1) > 0).all():
        raise ValueError("A review has no token assigned to a sentence.")
    labels, metadata, visible_targets = aux_task_labels(frame, cfg), [], []
    for i, (document, aligned) in enumerate(zip(frame.sentence_document, frame.evidence_alignment)):
        targets, item = aux_visible_evidence(document, aligned, offsets[i], sentence_ids[i], cfg["max_sentences"])
        visible_targets.append(targets)
        metadata.append(item)
    labels["evidence"] = np.stack(visible_targets)
    return AuxiliaryDataset(encoded, labels, sentence_ids, metadata)

class EvidenceHierarchicalClassifier(nn.Module):
    """Contextual sentences -> evidence-weighted aspect representations -> fusion + CLS.
    Aspect classifiers are siblings of the holistic pathway, not its inputs."""
    def __init__(self, base, aspects, dropout=0.1, attention_size=256, max_sentences=16):
        super().__init__()
        self.base = base
        self.aspects = tuple(aspects)
        self.max_sentences = max_sentences
        hidden_size = base.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.evidence_head = nn.Linear(hidden_size, len(self.aspects) * len(EVIDENCE_CLASSES))
        self.aspect_heads = nn.ModuleDict({a: nn.Linear(hidden_size, len(AUX_STATES)) for a in self.aspects})
        self.fusion_scorer = nn.Sequential(nn.Linear(hidden_size, attention_size), nn.Tanh(), nn.Linear(attention_size, 1))
        self.primary_head = nn.Linear(hidden_size * 2, len(AUX_OVERALL))

    def forward(self, input_ids, attention_mask, sentence_ids, return_extras=False):
        hidden = self.base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        with torch.autocast(device_type=hidden.device.type, enabled=False):
            states = hidden.float()
            batch, _, _ = states.shape
            membership = nn.functional.one_hot(sentence_ids.clamp(min=0), self.max_sentences).float()
            membership = membership * ((sentence_ids >= 0) & attention_mask.bool()).unsqueeze(-1).float()               # (B, T, S)
            counts = membership.sum(1)                                                          # (B, S)
            present = counts > 0
            if not present.any(dim=1).all():
                raise ValueError("Every review needs a visible pooled sentence.")
            sentences = torch.einsum("bts,btd->bsd", membership, states) / counts.clamp(min=1).unsqueeze(-1)
            sentences = self.dropout(sentences)                                                 # (B, S, d)
            evidence = self.evidence_head(sentences).view(batch, self.max_sentences, len(self.aspects), len(EVIDENCE_CLASSES))
            log_probs = evidence.log_softmax(-1)
            mention = torch.logsumexp(log_probs[..., 1:], dim=-1)                               # (B, S, K) log P(evidence)
            attention_logits = mention.permute(0, 2, 1).masked_fill(~present.unsqueeze(1), torch.finfo(mention.dtype).min)
            alpha = attention_logits.softmax(-1)                                                # (B, K, S)
            pooled = torch.einsum("bks,bsd->bkd", alpha, sentences)                             # (B, K, d)
            result = {a: self.aspect_heads[a](pooled[:, k]) for k, a in enumerate(self.aspects)}
            fusion = self.fusion_scorer(pooled).squeeze(-1).softmax(-1)                          # (B, K)
            aspect_summary = (fusion.unsqueeze(-1) * pooled).sum(1)                             # (B, d)
            features = torch.cat([self.dropout(states[:, 0, :]), aspect_summary], dim=-1)
            result["overall"] = self.primary_head(features)
            result["evidence"] = evidence
            result["evidence_present"] = present
        if return_extras:
            return result, {"fusion": fusion, "sentence_attention": alpha, "evidence_pred": evidence.argmax(-1)}
        return result

def aux_fit(model, train_loader, val_loader, criteria, hp, cfg, device):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=hp["lr"],
        total_steps=hp["epochs"] * len(train_loader), pct_start=0.1)
    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and dtype == torch.float16)
    best, state, best_epoch, stalled = -1.0, None, -1, 0
    history = []
    for epoch in range(1, hp["epochs"] + 1):
        model.train()
        losses, per_task = [], {k: [] for k in criteria}
        for batch in train_loader:
            targets = {k: v.to(device) for k, v in batch["targets"].items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
                logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["sentence_ids"].to(device))
                loss, terms = aux_loss(logits, targets, criteria, cfg["aux_strength"], cfg["evidence_strength"])
            checked_optimizer_step(loss, model, optimizer, scaler, 1.0, scheduler)
            losses.append(loss.item())
            for task, value in terms.items():
                per_task[task].append(value.item())
        probs, labels, _ = aux_predict(model, val_loader, device)
        score = aux_selection(probs, labels)
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "selection_f1": score,
               "task_losses": {k: (float(np.mean(v)) if v else None) for k, v in per_task.items()},
               "validation": {k: aux_metrics(labels[k], p.argmax(1), k == "overall") for k, p in probs.items()}}
        history.append(row)
        print(f"Epoch {epoch}/{hp['epochs']} | loss={row['train_loss']:.4f} | validation selection F1={score:.4f}", flush=True)
        if score > best:
            require_finite_model(model)
            best, best_epoch, stalled = score, epoch, 0
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stalled += 1
            if stalled >= cfg["patience"]:
                break
    if state is None:
        raise RuntimeError("No valid checkpoint.")
    model.load_state_dict(state)
    return best, best_epoch, history

def aux_tensor_identity(value):
    """Include tensor shape/type so different layouts cannot share a byte digest."""
    value = torch.as_tensor(value).detach().cpu().contiguous()
    return {"shape": list(value.shape), "dtype": str(value.dtype),
            "sha256": hashlib.sha256(value.numpy().tobytes()).hexdigest()}

def aux_tokenization_identity(datasets, cfg, model_name):
    # TRAIN and validation encodings both affect checkpoint/trial selection.
    partitions = {}
    for partition in ("train", "val"):
        dataset = datasets[partition]
        partitions[partition] = {
            "input_ids": aux_tensor_identity(dataset.encoded["input_ids"]),
            "attention_mask": aux_tensor_identity(dataset.encoded["attention_mask"]),
            "sentence_ids": aux_tensor_identity(dataset.sentence_ids),
            "visible_evidence_targets": aux_tensor_identity(dataset.labels["evidence"]),
        }
    return {"tokenizer": AUX_BACKBONES[model_name], "max_length": cfg["max_length"],
            "max_sentences": cfg["max_sentences"], "alignment_version": EVIDENCE_ALIGNMENT_VERSION,
            "runtime_sha256": AUX_RUNTIME_SHA256, "partitions": partitions}

def aux_verify_tokenization_identity(model_output, identity):
    """Check before saving anything that can be reused by an unfinished run."""
    model_output = Path(model_output)
    identity_path = model_output / "tokenization_identity.json"
    if identity_path.exists():
        saved = json.loads(identity_path.read_text())
        if saved != identity:
            raise ValueError("Tokenization or visible evidence changed. Keep the original run; use a new output version to retrain.")
        return saved
    has_training_state = (any(model_output.glob("trial_*.json"))
                          or any(model_output.glob("best_model*")))
    if has_training_state:
        raise ValueError("Existing trial/checkpoint lacks tokenization identity; refusing an unverifiable resume.")
    aux_json(identity_path, identity)
    return identity

def prepare_model(model_name, frames, cfg, output, device):
    """Tokenize this model's training/validation text, or verify its completed run."""
    from transformers import AutoTokenizer
    if model_name not in cfg["backbones"]:
        print(model_name, "is not selected in BACKBONES; skipping.")
        return None
    model_output = output / model_name
    model_output.mkdir(exist_ok=True)
    done = model_output / "complete.json"
    run = {"name": model_name, "output": model_output, "done": done,
           "complete": done.exists()}
    if done.exists():
        aux_verify_completed_run(model_output)
        print(model_name, "already complete and verified; skipping.")
        return run
    tokenizer = AutoTokenizer.from_pretrained(AUX_BACKBONES[model_name])
    datasets = {}
    # Test/audit predictions are made after selecting the model on validation data.
    for partition in ("train", "val"):
        datasets[partition] = aux_encode(tokenizer, frames[partition], cfg)
    loader_args = {"batch_size": cfg["batch_size"], "num_workers": cfg["num_workers"],
                   "pin_memory": device.type == "cuda"}
    val_loader = DataLoader(datasets["val"], shuffle=False,
        generator=torch.Generator().manual_seed(cfg["seed"] + 100_000), **loader_args)
    # Never overwrite provenance or class weights before validating an unfinished run.
    aux_verify_tokenization_identity(model_output, aux_tokenization_identity(datasets, cfg, model_name))
    visible_train_labels = {k: v.numpy() for k, v in datasets["train"].labels.items()}
    criteria, weights = aux_criteria(visible_train_labels, device)
    aux_json(model_output / "train_class_weights.json", weights)
    run.update(tokenizer=tokenizer, datasets=datasets, loader_args=loader_args,
               val_loader=val_loader, criteria=criteria)
    return run

def evaluate_saved_model(run, frames, cfg, best_record, device):
    """Load the selected model, evaluate each partition and save the existing outputs."""
    from transformers import AutoModel
    if run is None:
        return pd.DataFrame(columns=["model", "task", "partition"])
    if run["complete"]:
        aux_verify_completed_run(run["output"])
        return pd.read_csv(run["output"] / "metrics.csv")
    model_name, model_output, done = run["name"], run["output"], run["done"]
    tokenizer, datasets = run["tokenizer"], run["datasets"]
    loader_args = run["loader_args"]
    payload = torch.load(model_output / "best_model.pth", map_location="cpu", weights_only=True)
    if payload["record"] != best_record:
        raise ValueError("Selected checkpoint does not match completed trial records.")
    model = EvidenceHierarchicalClassifier(AutoModel.from_pretrained(AUX_BACKBONES[model_name]),
        cfg["aspects"], dropout=best_record["hp"]["dropout"], attention_size=cfg["attention_size"], max_sentences=cfg["max_sentences"])
    model.load_state_dict(payload["state_dict"], strict=True)
    require_finite_model(model)
    model.to(device)
    rows = []
    for partition in ("val", "test", "audit"):
        if partition not in datasets:
            datasets[partition] = aux_encode(tokenizer, frames[partition], cfg)
        loader = DataLoader(datasets[partition], shuffle=False,
            generator=torch.Generator().manual_seed(cfg["seed"] + 200_000), **loader_args)
        rows += aux_export(model, loader, frames[partition], cfg, model_name, partition, model_output, device)
    pd.DataFrame(rows).to_csv(model_output / "metrics.csv", index=False)
    tokenizer.save_pretrained(model_output / "tokenizer")
    model.base.config.save_pretrained(model_output / "encoder_config")
    aux_json(model_output / "selected_parameters.json", best_record)
    print(pd.DataFrame(rows).query("partition == 'test'").to_string(index=False))
    artifacts = {p.relative_to(model_output).as_posix(): aux_hash(p)
                 for p in model_output.rglob("*") if p.is_file() and p != done}
    aux_json(done, {"artifacts": artifacts})
    del model, loader
    run.pop("datasets", None)
    run.pop("val_loader", None)
    run["complete"] = True
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)


## 3.4 ALBERT: Evidence-supervised training

The following three steps prepare text (with sentence ids), search the existing parameter range and evaluate the selected model. Completed models and trials are reused as before; restarting Colab alone does not force retraining.


In [17]:
print("\n" + "=" * 60)
print('ALBERT' + " — DrugReview evidence-supervised hierarchical training")
print("=" * 60)



ALBERT — DrugReview evidence-supervised hierarchical training


### 3.4.1 Pre-tokenize

In [18]:
albert_run = prepare_model('albert', frames, AUX_CONFIG, output, device)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### 3.4.2 Hyperparameter tuning

In [19]:
best_record_albert = None
if albert_run is not None and not albert_run["complete"]:
    for index, hp in enumerate(plan):
        trial_path = albert_run["output"] / f"trial_{index + 1}.json"
        # A saved completed trial can be reused on interruption; the selected checkpoint is always updated first.
        if trial_path.exists():
            record = json.loads(trial_path.read_text())
            if record["hp"] != hp:
                raise ValueError("Saved trial plan differs.")
            if best_record_albert is None or record["score"] > best_record_albert["score"]:
                best_record_albert = record
            continue
        aux_seed(hp["seed"])
        model = EvidenceHierarchicalClassifier(AutoModel.from_pretrained(AUX_BACKBONES['albert']),
            AUX_CONFIG["aspects"], dropout=hp["dropout"], attention_size=AUX_CONFIG["attention_size"], max_sentences=AUX_CONFIG["max_sentences"])
        # Reset after head construction: identical data order and dropout RNG across paired conditions.
        aux_seed(hp["seed"])
        train_loader = DataLoader(albert_run["datasets"]["train"], shuffle=True,
            generator=torch.Generator().manual_seed(hp["seed"]), worker_init_fn=aux_worker_seed, **albert_run["loader_args"])
        print(f"albert trial {index + 1}/{len(plan)}", hp, flush=True)
        score, epoch, history = aux_fit(model, train_loader, albert_run["val_loader"], albert_run["criteria"], hp, AUX_CONFIG, device)
        record = {"trial": index + 1, "hp": hp, "score": score, "selected_epoch": epoch}
        if best_record_albert is None or score > best_record_albert["score"]:
            best_record_albert = record
            tmp = albert_run["output"] / "best_model.tmp"
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "record": record}, tmp)
            tmp.replace(albert_run["output"] / "best_model.pth")
        aux_json(albert_run["output"] / f"trial_{index + 1}_history.json", history)
        aux_json(trial_path, record)
        del model, train_loader
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

albert trial 1/10 {'lr': 1.160497696239576e-05, 'epochs': 6, 'dropout': 0.08535068283342875, 'seed': 2025}


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/6 | loss=2.8459 | validation selection F1=0.6605
Epoch 2/6 | loss=1.7868 | validation selection F1=0.7322
Epoch 3/6 | loss=1.5148 | validation selection F1=0.7388
Epoch 4/6 | loss=1.3243 | validation selection F1=0.7577
Epoch 5/6 | loss=1.2000 | validation selection F1=0.7640
Epoch 6/6 | loss=1.1364 | validation selection F1=0.7658
albert trial 2/10 {'lr': 1.1229524450717097e-05, 'epochs': 7, 'dropout': 0.09705888652849565, 'seed': 2026}
Epoch 1/7 | loss=2.8938 | validation selection F1=0.7165
Epoch 2/7 | loss=1.8230 | validation selection F1=0.7089
Epoch 3/7 | loss=1.5335 | validation selection F1=0.7450
Epoch 4/7 | loss=1.3413 | validation selection F1=0.7538
Epoch 5/7 | loss=1.2008 | validation selection F1=0.7662
Epoch 6/7 | loss=1.1005 | validation selection F1=0.7651
Epoch 7/7 | loss=1.0522 | validation selection F1=0.7691
albert trial 3/10 {'lr': 1.3271033162649868e-05, 'epochs': 9, 'dropout': 0.195692122900145, 'seed': 2027}
Epoch 1/9 | loss=3.0045 | validation selectio

### 3.4.3 Model evaluation

In [20]:
albert_results = evaluate_saved_model(
    albert_run, frames, AUX_CONFIG, best_record_albert, device
)
print(albert_results.query("partition == 'test'").to_string(index=False))


 model               variant              task partition                          reference_source  human_agreement      n  accuracy  weighted_f1  macro_f1     mae      qwk  evidence_strength  known_positive_n  known_negative_n  unknown_sentence_slots_n  known_target_mention_f1  known_target_mention_precision  known_target_mention_recall  selected_positive_direction_accuracy  localization_reviews_n  localization_excluded_reviews_n  top1_selected_quote_hit  selected_quote_attention_mass
albert evidence_hierarchical          efficacy      test                               GPT-4o-mini            False  20298  0.773919     0.787281  0.716850     NaN      NaN                NaN               NaN               NaN                       NaN                      NaN                             NaN                          NaN                                   NaN                     NaN                              NaN                      NaN                            NaN
albert evidence_hi

## 3.5 BioBERT: Evidence-supervised training

The following three steps prepare text (with sentence ids), search the existing parameter range and evaluate the selected model. Completed models and trials are reused as before; restarting Colab alone does not force retraining.


In [21]:
print("\n" + "=" * 60)
print('BioBERT' + " — DrugReview evidence-supervised hierarchical training")
print("=" * 60)



BioBERT — DrugReview evidence-supervised hierarchical training


### 3.5.1 Pre-tokenize

In [22]:
biobert_run = prepare_model('biobert', frames, AUX_CONFIG, output, device)


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

### 3.5.2 Hyperparameter tuning

In [23]:
best_record_biobert = None
if biobert_run is not None and not biobert_run["complete"]:
    for index, hp in enumerate(plan):
        trial_path = biobert_run["output"] / f"trial_{index + 1}.json"
        # A saved completed trial can be reused on interruption; the selected checkpoint is always updated first.
        if trial_path.exists():
            record = json.loads(trial_path.read_text())
            if record["hp"] != hp:
                raise ValueError("Saved trial plan differs.")
            if best_record_biobert is None or record["score"] > best_record_biobert["score"]:
                best_record_biobert = record
            continue
        aux_seed(hp["seed"])
        model = EvidenceHierarchicalClassifier(AutoModel.from_pretrained(AUX_BACKBONES['biobert']),
            AUX_CONFIG["aspects"], dropout=hp["dropout"], attention_size=AUX_CONFIG["attention_size"], max_sentences=AUX_CONFIG["max_sentences"])
        # Reset after head construction: identical data order and dropout RNG across paired conditions.
        aux_seed(hp["seed"])
        train_loader = DataLoader(biobert_run["datasets"]["train"], shuffle=True,
            generator=torch.Generator().manual_seed(hp["seed"]), worker_init_fn=aux_worker_seed, **biobert_run["loader_args"])
        print(f"biobert trial {index + 1}/{len(plan)}", hp, flush=True)
        score, epoch, history = aux_fit(model, train_loader, biobert_run["val_loader"], biobert_run["criteria"], hp, AUX_CONFIG, device)
        record = {"trial": index + 1, "hp": hp, "score": score, "selected_epoch": epoch}
        if best_record_biobert is None or score > best_record_biobert["score"]:
            best_record_biobert = record
            tmp = biobert_run["output"] / "best_model.tmp"
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "record": record}, tmp)
            tmp.replace(biobert_run["output"] / "best_model.pth")
        aux_json(biobert_run["output"] / f"trial_{index + 1}_history.json", history)
        aux_json(trial_path, record)
        del model, train_loader
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

biobert trial 1/10 {'lr': 1.160497696239576e-05, 'epochs': 6, 'dropout': 0.08535068283342875, 'seed': 2025}


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1/6 | loss=2.9669 | validation selection F1=0.6348
Epoch 2/6 | loss=1.9159 | validation selection F1=0.7192
Epoch 3/6 | loss=1.6232 | validation selection F1=0.7293
Epoch 4/6 | loss=1.4500 | validation selection F1=0.7381
Epoch 5/6 | loss=1.3431 | validation selection F1=0.7436
Epoch 6/6 | loss=1.3013 | validation selection F1=0.7435
biobert trial 2/10 {'lr': 1.1229524450717097e-05, 'epochs': 7, 'dropout': 0.09705888652849565, 'seed': 2026}
Epoch 1/7 | loss=3.0353 | validation selection F1=0.6572
Epoch 2/7 | loss=1.9366 | validation selection F1=0.7139
Epoch 3/7 | loss=1.6346 | validation selection F1=0.7398
Epoch 4/7 | loss=1.4514 | validation selection F1=0.7404
Epoch 5/7 | loss=1.3276 | validation selection F1=0.7506
Epoch 6/7 | loss=1.2576 | validation selection F1=0.7468
Epoch 7/7 | loss=1.2216 | validation selection F1=0.7509
biobert trial 3/10 {'lr': 1.3271033162649868e-05, 'epochs': 9, 'dropout': 0.195692122900145, 'seed': 2027}
Epoch 1/9 | loss=3.1175 | validation select

### 3.5.3 Model Evaluation

In [24]:
biobert_results = evaluate_saved_model(
    biobert_run, frames, AUX_CONFIG, best_record_biobert, device
)
print(biobert_results.query("partition == 'test'").to_string(index=False))


  model               variant              task partition                          reference_source  human_agreement      n  accuracy  weighted_f1  macro_f1      mae      qwk  evidence_strength  known_positive_n  known_negative_n  unknown_sentence_slots_n  known_target_mention_f1  known_target_mention_precision  known_target_mention_recall  selected_positive_direction_accuracy  localization_reviews_n  localization_excluded_reviews_n  top1_selected_quote_hit  selected_quote_attention_mass
biobert evidence_hierarchical          efficacy      test                               GPT-4o-mini            False  20298  0.771357     0.786126  0.716404      NaN      NaN                NaN               NaN               NaN                       NaN                      NaN                             NaN                          NaN                                   NaN                     NaN                              NaN                      NaN                            NaN
biobert eviden

## 3.6 Save Prediction

Each strength has a separate output version. Review predictions include aspect fusion weights and each aspect's most attended **visible** sentence span, text, truncation flag, and selected-quote hit indicator. Audit JSONL retains all visible sentence/token spans, annotation masks, coverage diagnostics and attention. Raw source sentence boundaries locate the source; their unobserved text is not presented as model input.

Evidence classification metrics apply only to known positive/negative targets. Localization metrics have their own eligible-review counts, and per-partition coverage files report unmatched/ambiguous quotes, token truncation, sentence-limit exclusions and unknown targets. Test and reserved human-audit predictions remain separate; GPT-reference metrics do not establish human validity.


In [25]:
all_model_metrics = pd.concat([albert_results, biobert_results], ignore_index=True)
summary_path = output / "all_models_metrics.csv"
all_model_metrics.to_csv(summary_path, index=False)
print(f"Saved model summary → {summary_path}")
for model_name in AUX_CONFIG["backbones"]:
    for partition in ("val", "test", "audit"):
        path = output / model_name / f"{model_name}_{partition}_predictions.csv"
        print(f"{model_name} {partition} predictions → {path}")
    fusion_path = output / model_name / f"{model_name}_fusion_by_holistic_class.csv"
    if fusion_path.exists():
        print(f"{model_name} mean fusion weights by holistic class → {fusion_path}")
print(all_model_metrics.query("partition == 'test'").to_string(index=False))
evidence_rows = all_model_metrics[all_model_metrics.task.str.startswith("evidence_") & all_model_metrics.partition.eq("test")]
if len(evidence_rows):
    print("\nSentence-level evidence (test):")
    print(evidence_rows[["model", "task", "n", "known_positive_n", "known_negative_n", "macro_f1", "localization_reviews_n", "top1_selected_quote_hit", "selected_quote_attention_mass"]].to_string(index=False))


Saved model summary → /content/drive/MyDrive/NLP_Projects/03_DrugReview/07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1/all_models_metrics.csv
albert val predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1/albert/albert_val_predictions.csv
albert test predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1/albert/albert_test_predictions.csv
albert audit predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1/albert/albert_audit_predictions.csv
albert mean fusion weights by holistic class → /content/drive/MyDrive/NLP_Projects/03_DrugReview/07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1/albert/albert_fusion_by_holistic_class.csv
biobert val predictions → /content/drive/MyDrive/NLP_Projects/03_DrugReview/07_Evidence